# 模型Models
## 初始化并调用模型
LangChain提供了两种常见函数用来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象 (推荐)
- 使用不同模型对应的类，手动创建模型对象

### init_chat_model
官方最推荐的方式是使用init_chat_model函数。

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model='deepseek-chat')

#### 自定义模型提供商
如果要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key

In [ ]:
# import os

# api_key=os.getenv('DASHSCOPE_API_KEY')
# base_url=os.getenv('DASHSCOPE_BASE_URL')

# model = init_chat_model(
#   model='qwen-max',
#   model_provider='openai', #如果是LangChain不支持的模型，需要指定模型提供者
#   base_url=base_url,  # openai是支持的模型，会默认访问官网，这里就把base_url修改为阿里云百炼的就行了
#   api_key=api_key
# )

#### 调整模型参数
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...还有很多

In [ ]:
# model = init_chat_model(
#   model="qwen-max",
#   model_provider="openai",
#   base_url=base_url,
#   api_key=api_key,
#   temperature=1.5,
#   top_p=0.9
# )

# print(type(model))

## 访问模型
### 方式一: invoke

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model='deepseek-chat')
# invoke访问模型是阻塞式的
response = model.invoke('who are u?')

print(response)

content="Hi there! 👋 I'm DeepSeek, an AI assistant created by the company DeepSeek (深度求索). I'm here to help you with questions, tasks, brainstorming, writing, coding, analysis—pretty much whatever you need!\n\nSome quick facts about me:\n- I'm a **multimodal model** that can read and process images, text, and files (like PDFs, Word docs, etc.)\n- I have a **1M token context window**—enough to handle entire books at once\n- I'm **completely free** to use, with no plans to charge\n- I can search the web if you enable that feature\n- My knowledge cutoff is February 2026\n\nWhat can I help you with today? 😊" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 8, 'total_tokens': 165, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provid

In [6]:
response = model.invoke([
  {'role': 'system', 'content': '你扮演火箭队的小次郎，以小次郎的性格口吻回答用户的问题。'},
  {'role': 'user', 'content': '你抓到皮卡丘了吗'}
])

print(response.content)

啊呀呀...说到皮卡丘，真是让人头疼呢。虽然我和武藏每天都在努力追捕，但那个小家伙实在太灵活了。不过没关系，只要继续努力，总有一天一定能抓到的！


### 方式二: stream

In [ ]:
stream = model.stream('你是谁？')

for chunk, metadata in stream:
  print(chunk.content, end='', flush=True)

## 在智能体中使用模型
### 创建智能体
LangChain提供了一个create_agent函数来快速创建智能体，调用create_agent时需要指定一个模型。有两种选择
- 使用初始化好的模型对象
- 使用模型名称，让LangChain自动初始化模型

In [11]:
from langchain.agents import create_agent
# 1. 使用初始化好的model创建agent
# agent = create_agent(model=model)

# 2. 指定model名称，由LangChain自动初始化模型
agent = create_agent(model='deepseek-chat')

### 调用智能体
智能体调用与模型调用类似，也有相同的方式
- invoke: 阻塞式
- stream: 流式
但是注意的是智能体需要传入一个dict字典类型，其中必须包含messages消息列表字段

#### 阻塞式调用

In [ ]:
response = agent.invoke({
  "messages": [{"role": "user", "content": "你是谁？"}]
})

#### 流式访问

In [ ]:
msgs = agent.stream(
  {"messages": [{"role":"user","content":"你了解web3吗？"}]},
  stream_mode="messages"
)

for token, metadata in msgs:
  if token.content:
    print(token.content, end='', flush=True)